# Supply chain and Logistic Analytics

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('DataCoSupplyChainDataset.csv',
    encoding='latin1'
)

In [3]:
df

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.750000,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.750000,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.750000,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.750000,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.750000,0,1/15/2018 11:24,Standard Class
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180514,CASH,4,4,40.000000,399.980011,Shipping on time,0,45,Fishing,Brooklyn,...,NaN,1004,45,NaN,http://images.acmesports.sports/Field+%26+Stre...,Field & Stream Sportsman 16 Gun Fire Safe,399.980011,0,1/20/2016 3:40,Standard Class
180515,DEBIT,3,2,-613.770019,395.980011,Late delivery,1,45,Fishing,Bakersfield,...,NaN,1004,45,NaN,http://images.acmesports.sports/Field+%26+Stre...,Field & Stream Sportsman 16 Gun Fire Safe,399.980011,0,1/19/2016 1:34,Second Class
180516,TRANSFER,5,4,141.110001,391.980011,Late delivery,1,45,Fishing,Bristol,...,NaN,1004,45,NaN,http://images.acmesports.sports/Field+%26+Stre...,Field & Stream Sportsman 16 Gun Fire Safe,399.980011,0,1/20/2016 21:00,Standard Class
180517,PAYMENT,3,4,186.229996,387.980011,Advance shipping,0,45,Fishing,Caguas,...,NaN,1004,45,NaN,http://images.acmesports.sports/Field+%26+Stre...,Field & Stream Sportsman 16 Gun Fire Safe,399.980011,0,1/18/2016 20:18,Standard Class


In [4]:
drop_cols = [
    'Customer Email', 'Customer Password', 'Customer Street',
    'Product Description', 'Product Image',
    'Order Item Cardprod Id', 'Order Customer Id',
    'Customer Zipcode', 'Order Zipcode',
    'Benefit per order',
    'Sales per customer',
    'Product Status',   
]
df.drop(columns=drop_cols, inplace=True)

In [5]:
rename_map = {
    'Type':                        'payment_type',
    'Days for shipping (real)':    'days_shipping_real',
    'Days for shipment (scheduled)':'days_shipping_scheduled',
    'Delivery Status':             'delivery_status',
    'Late_delivery_risk':          'late_delivery_risk',
    'Category Id':                 'category_id',
    'Category Name':               'category_name',
    'Customer City':               'customer_city',
    'Customer Country':            'customer_country',
    'Customer Fname':              'customer_fname',
    'Customer Id':                 'customer_id',
    'Customer Lname':              'customer_lname',
    'Customer Segment':            'customer_segment',
    'Customer State':              'customer_state',
    'Department Id':               'department_id',
    'Department Name':             'department_name',
    'Latitude':                    'latitude',
    'Longitude':                   'longitude',
    'Market':                      'market',
    'Order City':                  'order_city',
    'Order Country':               'order_country',
    'order date (DateOrders)':     'order_date',
    'Order Id':                    'order_id',
    'Order Item Discount':         'item_discount',
    'Order Item Discount Rate':    'item_discount_rate',
    'Order Item Id':               'order_item_id',
    'Order Item Product Price':    'item_product_price',
    'Order Item Profit Ratio':     'item_profit_ratio',
    'Order Item Quantity':         'item_quantity',
    'Sales':                       'sales',
    'Order Item Total':            'order_item_total',
    'Order Profit Per Order':      'order_profit',
    'Order Region':                'order_region',
    'Order State':                 'order_state',
    'Order Status':                'order_status',
    'Product Card Id':             'product_id',
    'Product Category Id':         'product_category_id',
    'Product Name':                'product_name',
    'Product Price':               'product_price',
    'shipping date (DateOrders)':   'shipping_date',
    'Shipping Mode':               'shipping_mode',
}
df.rename(columns=rename_map, inplace=True)

In [11]:
# ── 4. FIX DATA TYPES ──────────────────────────────────────
df['order_date']    = pd.to_datetime(df['order_date'],    errors='coerce')
df['shipping_date'] = pd.to_datetime(df['shipping_date'], errors='coerce')

In [12]:
# ── 5. FEATURE ENGINEERING ─────────────────────────────────

# A. Delivery delay in days (positive = late)
df['delivery_delay_days'] = (
    df['days_shipping_real'] - df['days_shipping_scheduled']
)

# B. On-time flag (1 = on time or early)
df['is_on_time'] = (df['delivery_delay_days'] <= 0).astype(int)

# C. Discount flag (1 = discounted order)
df['is_discounted'] = (df['item_discount_rate'] > 0).astype(int)

# D. Profit flag (1 = profitable, 0 = loss)
df['is_profitable'] = (df['order_profit'] > 0).astype(int)

# E. Revenue after discount
df['net_revenue'] = df['sales'] - df['item_discount']

# F. Date parts for dim_date
df['order_year']    = df['order_date'].dt.year
df['order_month']   = df['order_date'].dt.month
df['order_quarter'] = df['order_date'].dt.quarter
df['order_month_name'] = df['order_date'].dt.strftime('%B')

# G. Shipping duration (order → ship)
df['processing_days'] = (
    df['shipping_date'] - df['order_date']
).dt.days

In [14]:
# ── 6. NULL HANDLING ───────────────────────────────────────

df['customer_lname']=df['customer_lname'].fillna('')

In [17]:
# ── 7. EXPORT CLEANED FILE ─────────────────────────────────
df.to_csv('D:/SC & LA/dataco_clean.csv', index=False)
print(f"Clean file: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("Nulls remaining:", df.isnull().sum().sum())

Clean file: 180,519 rows × 51 columns
Nulls remaining: 0


In [18]:
df.isnull().sum()

payment_type               0
days_shipping_real         0
days_shipping_scheduled    0
delivery_status            0
late_delivery_risk         0
category_id                0
category_name              0
customer_city              0
customer_country           0
customer_fname             0
customer_id                0
customer_lname             0
customer_segment           0
customer_state             0
department_id              0
department_name            0
latitude                   0
longitude                  0
market                     0
order_city                 0
order_country              0
order_date                 0
order_id                   0
item_discount              0
item_discount_rate         0
order_item_id              0
item_product_price         0
item_profit_ratio          0
item_quantity              0
sales                      0
order_item_total           0
order_profit               0
order_region               0
order_state                0
order_status  